In [2]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

In [3]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

In [73]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("CUSTOM_hg38_episign/meth_matrix.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
epiSize = {}
for epiSign in [x for x in X.columns if x not in ('coord')]:
    epiSize[epiSign] = sum(X[epiSign] > 99)  # Catch 100% methyl

In [74]:
# Transpose
X_t = X.T  # Required
print(X_t.index)

to_PCA = X_t
if 'epiSize' in X_t.columns:
    to_PCA = X_t.drop('epiSize', axis=1)

# Remove 2nd row = size of epiSignormalize) then normalize
X_scaled = StandardScaler().fit_transform(to_PCA)

Index(['ADCADN.bed', 'ATRX.bed', 'AUTS18.bed', 'BAFopathy.bed', 'BFLS.bed',
       'CHARGE.bed', 'CdLS.bed', 'Down.bed', 'Dup7.bed', 'EEOC.bed',
       'FLHS.bed', 'GTPTS.bed', 'HMA.bed', 'HVDAS_C.bed', 'HVDAS_T.bed',
       'ICF1.bed', 'ICF2_3_4.bed', 'KDVS.bed', 'Kabuki.bed', 'Kleefstra.bed',
       'MRD51.bed', 'MRX93.bed', 'MRX97.bed', 'MRXCJS.bed', 'MRXSN.bed',
       'MRXSSR.bed', 'RMNS.bed', 'RSTS.bed', 'SBBYSS.bed', 'SETD1B.bed',
       'Sotos.bed', 'TBRS.bed', 'WDSTS.bed', 'Williams.bed', 'HG002_combined',
       'barcode04_combined'],
      dtype='object')


In [75]:
# Run PCA:
NB_COMPON = 3
pca = PCA(n_components=NB_COMPON)
pcs = pca.fit_transform(X_scaled)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

In [76]:
# Top 3 features of each componennt
compon_0_top = np.abs(pca.components_[0]).argsort()[::-1][:3]
print("Component 0:", X.index[compon_0_top])

compon_1_top = np.abs(pca.components_[1]).argsort()[::-1][:3]
print("Component 1:", X.index[compon_1_top])

Component 0: Index(['11:121157882-121157883', '10:45443215-45443216',
       '4:122379484-122379485'],
      dtype='object', name='coord')
Component 1: Index(['3:45795704-45795705', '3:42160156-42160157', '17:74763142-74763143'], dtype='object', name='coord')


In [77]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
print(pcs_df.head())
print("\nCoords of samples:")
print(pcs_df.loc['barcode04_combined'])
print(pcs_df.loc['HG002_combined'])

                compon0   compon1   compon2
ADCADN.bed    -8.163853 -2.716288 -1.427895
ATRX.bed      -1.275438  0.022436 -1.321275
AUTS18.bed    -2.606411 -1.052068 -1.349958
BAFopathy.bed -2.268899 -9.619868 -9.510190
BFLS.bed      -2.445427 -0.333894 -3.540970

Coords of samples:
compon0    25.251140
compon1   -13.122593
compon2    -1.760406
Name: barcode04_combined, dtype: float64
compon0    90.481875
compon1     4.959419
compon2     0.964793
Name: HG002_combined, dtype: float64


In [78]:
# Plot PCA
x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=epiSize,
        labels={x_compon:':'.join([x_compon,dict_compon[x_compon]]), y_compon:':'.join([y_compon,dict_compon[y_compon]])}
)
fig.show()